# Datathon Passos Mágicos — Tratamento da Base PEDE (2022-2024)

**Pós-graduação em Data Analytics — FIAP Postech — Fase 5**

Objetivo deste notebook: carregar a base `BASE_DE_DADOS_PEDE_2024_-_DATATHON.xlsx`,
diagnosticar problemas de qualidade e consolidar as 3 abas anuais (`PEDE2022`, `PEDE2023`,
`PEDE2024`) em um **painel longo** (1 linha = aluno-ano), pronto para alimentar a análise
exploratória (11 perguntas de negócio) e o modelo preditivo de risco de defasagem.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 60)

CAMINHO = "/mnt/user-data/uploads/BASE_DE_DADOS_PEDE_2024_-_DATATHON.xlsx"

xl = pd.ExcelFile(CAMINHO)
df22 = xl.parse("PEDE2022")
df23 = xl.parse("PEDE2023")
df24 = xl.parse("PEDE2024")

print("PEDE2022:", df22.shape)
print("PEDE2023:", df23.shape)
print("PEDE2024:", df24.shape)


PEDE2022: (860, 42)
PEDE2023: (1014, 48)
PEDE2024: (1156, 50)


## 1. Diagnóstico da base

Cada aba é uma "foto" anual da pesquisa PEDE, com nomenclatura de colunas que **muda
de um ano para o outro** (ex.: `Matem` em 2022 vira `Mat` em 2023/2024). A chave `RA`
(Registro do Aluno) é consistente entre os anos e permite montar o painel longitudinal.


In [2]:
# Sobreposição de alunos entre anos (confirma que é o mesmo grupo, com entrada/saída natural)
ra22, ra23, ra24 = set(df22['RA']), set(df23['RA']), set(df24['RA'])
print("Alunos únicos 2022:", len(ra22))
print("Alunos únicos 2023:", len(ra23))
print("Alunos únicos 2024:", len(ra24))
print("Interseção 2022∩2023:", len(ra22 & ra23))
print("Interseção 2023∩2024:", len(ra23 & ra24))
print("Interseção 2022∩2024:", len(ra22 & ra24))


Alunos únicos 2022: 860
Alunos únicos 2023: 1014
Alunos únicos 2024: 1156
Interseção 2022∩2023: 600
Interseção 2023∩2024: 765
Interseção 2022∩2024: 472


In [3]:
# Problema 1: codificação de Fase inconsistente entre anos
print("Fase 2022:", sorted(df22['Fase'].unique(), key=str))
print("Fase 2023:", sorted(df23['Fase'].unique(), key=str))
print("Fase 2024 (amostra):", sorted(df24['Fase'].astype(str).unique())[:15], "...")


Fase 2022: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
Fase 2023: ['ALFA', 'FASE 1', 'FASE 2', 'FASE 3', 'FASE 4', 'FASE 5', 'FASE 6', 'FASE 7', 'FASE 8']
Fase 2024 (amostra): ['1A', '1B', '1C', '1D', '1E', '1G', '1H', '1J', '1K', '1L', '1M', '1N', '1P', '1R', '2A'] ...


In [4]:
# Problema 2: registros com cadastro pendente em 2024 (INDE = 'INCLUIR')
pendentes = df24[df24['INDE 2024'].astype(str) == 'INCLUIR']
print("Registros pendentes em 2024:", len(pendentes))
print(pendentes[['RA', 'Fase', 'Turma', 'Ativo/ Inativo']].head())


Registros pendentes em 2024: 38
           RA Fase Turma Ativo/ Inativo
1118  RA-1637    9     9       Cursando
1119  RA-1638    9     9       Cursando
1120  RA-1268    9     9       Cursando
1121  RA-1260    9     9       Cursando
1122  RA-1639    9     9       Cursando


In [5]:
# Problema 3: valores de Idade corrompidos (viraram data do Excel) na aba 2023
idade_invalida = df23[pd.to_numeric(df23['Idade'], errors='coerce').isna() & df23['Idade'].notna()]
print("Registros com Idade corrompida em 2023:", len(idade_invalida))
print(idade_invalida['Idade'].unique())


Registros com Idade corrompida em 2023: 399
[datetime.datetime(1900, 1, 8, 0, 0) datetime.datetime(1900, 1, 7, 0, 0)
 datetime.datetime(1900, 1, 11, 0, 0) datetime.datetime(1900, 1, 9, 0, 0)
 datetime.datetime(1900, 1, 10, 0, 0) datetime.datetime(1900, 1, 14, 0, 0)
 datetime.datetime(1900, 1, 13, 0, 0) datetime.datetime(1900, 1, 12, 0, 0)
 datetime.datetime(1900, 1, 15, 0, 0) datetime.datetime(1900, 1, 17, 0, 0)
 datetime.datetime(1900, 1, 16, 0, 0) datetime.datetime(1900, 1, 19, 0, 0)
 datetime.datetime(1900, 1, 18, 0, 0) datetime.datetime(1900, 1, 20, 0, 0)
 datetime.datetime(1900, 1, 21, 0, 0) datetime.datetime(1900, 1, 26, 0, 0)]


In [6]:
# Problema 4: colunas 100% vazias em 2023/2024 (a PM parou de preencher esses campos)
for nome, d in [('2023', df23), ('2024', df24)]:
    vazias = d.columns[d.isna().mean() == 1.0].tolist()
    print(f"Colunas 100% vazias em {nome}: {vazias}\n")


Colunas 100% vazias em 2023: ['Pedra 23', 'INDE 23', 'Cg', 'Cf', 'Ct', 'Rec Av1', 'Rec Av2', 'Rec Av3', 'Rec Av4', 'Rec Psicologia', 'Indicado', 'Atingiu PV', 'Destaque IEG', 'Destaque IDA', 'Destaque IPV', 'Destaque IPV.1']

Colunas 100% vazias em 2024: ['Cg', 'Cf', 'Ct', 'Rec Av1', 'Rec Av2', 'Rec Psicologia', 'Indicado', 'Atingiu PV', 'Destaque IEG', 'Destaque IDA', 'Destaque IPV']



## 2. Padronização e empilhamento (painel longo)

Mapeamos cada aba para um **schema canônico** de colunas (mesmo nome para o mesmo
conceito, independente do ano) e empilhamos as três em um único DataFrame, com a
coluna `Ano_Referencia` identificando o ano de cada observação.


In [7]:
MAPA_22 = {
    "Fase": "Fase_Raw", "Turma": "Turma", "Nome": "Nome",
    "Ano nasc": "Ano_Nascimento", "Idade 22": "Idade", "Gênero": "Genero",
    "Ano ingresso": "Ano_Ingresso", "Instituição de ensino": "Instituicao_Ensino",
    "INDE 22": "INDE", "Pedra 22": "Pedra",
    "Cg": "Cg", "Cf": "Cf", "Ct": "Ct", "Nº Av": "Num_Avaliacoes",
    "IAA": "IAA", "IEG": "IEG", "IPS": "IPS", "IDA": "IDA", "IPV": "IPV", "IAN": "IAN",
    "Rec Psicologia": "Rec_Psicologia",
    "Matem": "Nota_Matematica", "Portug": "Nota_Portugues", "Inglês": "Nota_Ingles",
    "Indicado": "Indicado_Bolsa", "Atingiu PV": "Atingiu_Ponto_Virada",
    "Fase ideal": "Fase_Ideal", "Defas": "Defasagem",
    "Destaque IEG": "Destaque_IEG", "Destaque IDA": "Destaque_IDA", "Destaque IPV": "Destaque_IPV",
}
MAPA_23 = {
    "Fase": "Fase_Raw", "INDE 2023": "INDE", "Pedra 2023": "Pedra", "Turma": "Turma",
    "Nome Anonimizado": "Nome", "Idade": "Idade", "Gênero": "Genero",
    "Ano ingresso": "Ano_Ingresso", "Instituição de ensino": "Instituicao_Ensino",
    "Cg": "Cg", "Cf": "Cf", "Ct": "Ct", "Nº Av": "Num_Avaliacoes",
    "IAA": "IAA", "IEG": "IEG", "IPS": "IPS", "IPP": "IPP", "IDA": "IDA", "IPV": "IPV", "IAN": "IAN",
    "Rec Psicologia": "Rec_Psicologia",
    "Mat": "Nota_Matematica", "Por": "Nota_Portugues", "Ing": "Nota_Ingles",
    "Indicado": "Indicado_Bolsa", "Atingiu PV": "Atingiu_Ponto_Virada",
    "Fase Ideal": "Fase_Ideal", "Defasagem": "Defasagem",
    "Destaque IEG": "Destaque_IEG", "Destaque IDA": "Destaque_IDA", "Destaque IPV": "Destaque_IPV",
}
MAPA_24 = {
    "Fase": "Fase_Raw", "INDE 2024": "INDE", "Pedra 2024": "Pedra", "Turma": "Turma",
    "Nome Anonimizado": "Nome", "Idade": "Idade", "Gênero": "Genero",
    "Ano ingresso": "Ano_Ingresso", "Instituição de ensino": "Instituicao_Ensino",
    "Escola": "Escola", "Ativo/ Inativo": "Status_Matricula",
    "Cg": "Cg", "Cf": "Cf", "Ct": "Ct", "Nº Av": "Num_Avaliacoes",
    "IAA": "IAA", "IEG": "IEG", "IPS": "IPS", "IPP": "IPP", "IDA": "IDA", "IPV": "IPV", "IAN": "IAN",
    "Rec Psicologia": "Rec_Psicologia",
    "Mat": "Nota_Matematica", "Por": "Nota_Portugues", "Ing": "Nota_Ingles",
    "Indicado": "Indicado_Bolsa", "Atingiu PV": "Atingiu_Ponto_Virada",
    "Fase Ideal": "Fase_Ideal", "Defasagem": "Defasagem",
    "Destaque IEG": "Destaque_IEG", "Destaque IDA": "Destaque_IDA", "Destaque IPV": "Destaque_IPV",
}

def preparar(df, mapa, ano):
    d = df.rename(columns=mapa).copy()
    manter = ["RA"] + list(dict.fromkeys(mapa.values()))
    manter = [c for c in manter if c in d.columns]
    d = d[manter]
    d["Ano_Referencia"] = ano
    return d

t22 = preparar(df22, MAPA_22, 2022)
t23 = preparar(df23, MAPA_23, 2023)
t24 = preparar(df24, MAPA_24, 2024)

painel = pd.concat([t22, t23, t24], ignore_index=True, sort=False)
print("Painel consolidado:", painel.shape)
painel.head()


Painel consolidado: (3030, 36)


,RA,Fase_Raw,Turma,Nome,Ano_Nascimento,Idade,Genero,Ano_Ingresso,Instituicao_Ensino,INDE,Pedra,Cg,Cf,Ct,Num_Avaliacoes,IAA,IEG,IPS,IDA,IPV,IAN,Rec_Psicologia,Nota_Matematica,Nota_Portugues,Nota_Ingles,Indicado_Bolsa,Atingiu_Ponto_Virada,Fase_Ideal,Defasagem,Destaque_IEG,Destaque_IDA,Destaque_IPV,Ano_Referencia,IPP,Escola,Status_Matricula
0,RA-1,7,A,Aluno-1,2003.0,19,Menina,2016,Escola Pública,5.783,Quartzo,753.0,18.0,10.0,4.0,8.3,4.1,5.6,4.0,7.278,5.0,Requer avaliação,2.7,3.5,6.0,Sim,Não,Fase 8 (Universitários),-1,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...,2022,NaN,NaN,NaN
1,RA-2,7,A,Aluno-2,2005.0,17,Menina,2017,Rede Decisão,7.055,Ametista,469.0,8.0,3.0,4.0,8.8,5.2,6.3,6.8,6.778,10.0,Sem limitações,6.3,4.5,9.7,Não,Não,Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...,2022,NaN,NaN,NaN
2,RA-3,7,A,Aluno-3,2005.0,17,Menina,2016,Rede Decisão,6.591,Ágata,629.0,13.0,6.0,4.0,0.0,7.9,5.6,5.6,7.556,10.0,Sem limitações,5.8,4.0,6.9,Não,Não,Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Destaque: A sua boa integração aos Princípios ...,2022,NaN,NaN,NaN
3,RA-4,7,A,Aluno-4,2005.0,17,Menino,2017,Rede Decisão,5.951,Quartzo,731.0,15.0,7.0,4.0,8.8,4.5,5.6,5.0,5.278,10.0,Requer avaliação,2.8,3.5,8.7,Não,Não,Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...,2022,NaN,NaN,NaN
4,RA-5,7,A,Aluno-5,2005.0,17,Menina,2016,Rede Decisão,7.427,Ametista,344.0,6.0,2.0,4.0,7.9,8.6,5.6,5.2,7.389,10.0,Requer avaliação,7.0,2.9,5.7,Não,Não,Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...,2022,NaN,NaN,NaN


## 3. Tratamentos de qualidade

- **INDE = "INCLUIR"** → convertido para nulo, com flag `Status_Cadastro` marcando o
  registro como `Pendente` (mantém rastreabilidade sem inventar um valor).
- **Fase = "9" nos mesmos registros pendentes** → nulo (não é uma fase real, é
  placeholder de cadastro em aberto).
- **Idade corrompida (datas do Excel)** → convertida para nulo (tratamento simples,
  sem tentar reconstruir o valor original).
- **Fase_Num** → nova coluna numérica, extraída de `Fase_Raw`, unificando `ALFA` = 0
  e removendo a letra da turma (ex.: `"1A"` → `1`), para permitir comparação de nível
  entre os três anos.


In [8]:
painel["Status_Cadastro"] = np.where(
    painel["INDE"].astype(str).str.upper() == "INCLUIR", "Pendente", "Completo"
)
painel["INDE"] = pd.to_numeric(painel["INDE"], errors="coerce")

painel.loc[painel["Status_Cadastro"] == "Pendente", "Fase_Raw"] = np.nan

painel["Idade"] = pd.to_numeric(painel["Idade"], errors="coerce")

def normaliza_fase(valor):
    if pd.isna(valor):
        return np.nan
    s = str(valor).upper().strip()
    if s == "ALFA":
        return 0
    digitos = "".join(ch for ch in s if ch.isdigit())
    return int(digitos) if digitos else np.nan

painel["Fase_Num"] = painel["Fase_Raw"].apply(normaliza_fase)
painel["Genero"] = painel["Genero"].astype(str).str.strip().str.title()

painel.head()


,RA,Fase_Raw,Turma,Nome,Ano_Nascimento,Idade,Genero,Ano_Ingresso,Instituicao_Ensino,INDE,Pedra,Cg,Cf,Ct,Num_Avaliacoes,IAA,IEG,IPS,IDA,IPV,IAN,Rec_Psicologia,Nota_Matematica,Nota_Portugues,Nota_Ingles,Indicado_Bolsa,Atingiu_Ponto_Virada,Fase_Ideal,Defasagem,Destaque_IEG,Destaque_IDA,Destaque_IPV,Ano_Referencia,IPP,Escola,Status_Matricula,Status_Cadastro,Fase_Num
0,RA-1,7,A,Aluno-1,2003.0,19.0,Menina,2016,Escola Pública,5.783,Quartzo,753.0,18.0,10.0,4.0,8.3,4.1,5.6,4.0,7.278,5.0,Requer avaliação,2.7,3.5,6.0,Sim,Não,Fase 8 (Universitários),-1,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...,2022,NaN,NaN,NaN,Completo,7.0
1,RA-2,7,A,Aluno-2,2005.0,17.0,Menina,2017,Rede Decisão,7.055,Ametista,469.0,8.0,3.0,4.0,8.8,5.2,6.3,6.8,6.778,10.0,Sem limitações,6.3,4.5,9.7,Não,Não,Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...,2022,NaN,NaN,NaN,Completo,7.0
2,RA-3,7,A,Aluno-3,2005.0,17.0,Menina,2016,Rede Decisão,6.591,Ágata,629.0,13.0,6.0,4.0,0.0,7.9,5.6,5.6,7.556,10.0,Sem limitações,5.8,4.0,6.9,Não,Não,Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Destaque: A sua boa integração aos Princípios ...,2022,NaN,NaN,NaN,Completo,7.0
3,RA-4,7,A,Aluno-4,2005.0,17.0,Menino,2017,Rede Decisão,5.951,Quartzo,731.0,15.0,7.0,4.0,8.8,4.5,5.6,5.0,5.278,10.0,Requer avaliação,2.8,3.5,8.7,Não,Não,Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...,2022,NaN,NaN,NaN,Completo,7.0
4,RA-5,7,A,Aluno-5,2005.0,17.0,Menina,2016,Rede Decisão,7.427,Ametista,344.0,6.0,2.0,4.0,7.9,8.6,5.6,5.2,7.389,10.0,Requer avaliação,7.0,2.9,5.7,Não,Não,Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...,2022,NaN,NaN,NaN,Completo,7.0


## 4. Conferência final e exportação

Antes de exportar, validamos volumetria por ano, distribuição do status de cadastro
e o percentual de nulos remanescente por coluna — para deixar claro, na EDA seguinte,
quais indicadores têm cobertura completa em quais anos.


In [9]:
print("Linhas (aluno-ano) por Ano_Referencia:")
print(painel["Ano_Referencia"].value_counts().sort_index())

print("\nStatus_Cadastro:")
print(painel["Status_Cadastro"].value_counts())

print("\n% de nulos por coluna:")
print((painel.isna().mean() * 100).round(1).sort_values(ascending=False))


Linhas (aluno-ano) por Ano_Referencia:
Ano_Referencia
2022     860
2023    1014
2024    1156
Name: count, dtype: int64

Status_Cadastro:
Status_Cadastro
Completo    2992
Pendente      38
Name: count, dtype: int64

% de nulos por coluna:


Ano_Nascimento          71.6
Destaque_IDA            71.6
Rec_Psicologia          71.6
Cg                      71.6
Ct                      71.6
Cf                      71.6
Destaque_IPV            71.6
Destaque_IEG            71.6
Atingiu_Ponto_Virada    71.6
Indicado_Bolsa          71.6
Nota_Ingles             64.0
Escola                  61.9
Status_Matricula        61.8
IPP                     34.3
Idade                   13.2
INDE                     6.1
Nota_Matematica          6.1
Nota_Portugues           6.1
IDA                      5.9
IPV                      5.9
IPS                      5.6
IAA                      5.4
Pedra                    4.9
IEG                      2.5
Num_Avaliacoes           2.5
Fase_Raw                 1.3
Fase_Num                 1.3
Turma                    0.0
RA                       0.0
Nome                     0.0
IAN                      0.0
Instituicao_Ensino       0.0
Genero                   0.0
Ano_Ingresso             0.0
Defasagem     

In [10]:
OUT_PATH = "/mnt/user-data/outputs/painel_pede_tratado.csv"
painel.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"Base tratada exportada para: {OUT_PATH}")
print(f"Shape final: {painel.shape}")


Base tratada exportada para: /mnt/user-data/outputs/painel_pede_tratado.csv
Shape final: (3030, 38)


## 5. Limitações conhecidas (importantes para a análise e o modelo)

- **IPP (Indicador Psicopedagógico) não existe na aba 2022** — só está disponível a
  partir de 2023. Qualquer análise que combine IPP com anos anteriores terá nulo em 2022.
- **`Cg`, `Cf`, `Ct`, `Atingiu_Ponto_Virada`, `Indicado_Bolsa`, `Rec_Psicologia`,
  `Destaque_*`** ficaram 100% vazios em 2023 e 2024 — a Passos Mágicos aparentemente
  parou de preencher esses campos nesse formato. Isso limita a pergunta 7 (comportamentos
  que influenciam o IPV) a uma análise mais robusta apenas em 2022.
- **38 registros de 2024 ficam com `Fase_Num` e `INDE` nulos** (status `Pendente`) —
  alunos ativos, mas sem avaliação lançada no momento da extração da base.
